In [15]:
import gzip
import re
from collections import Counter

In [2]:
jan_evidence_path = 'january_evidence.json.gz'
apr_evidence_path = 'april_evidence.json.gz'

In [ ]:
'''
{
  "alleleOrigins": [
    "germline"
  ],
  "datasourceId": "eva",
  "datatypeId": "genetic_association",
  "clinicalSignificances": [
    "uncertain significance"
  ],
  "confidence": "criteria provided, single submitter",
  "studyId": "RCV001155073",
  "releaseDate": "2020-05-31",
  "targetFromSourceId": "ENSG00000154124",
  "variantFunctionalConsequenceId": "SO_0001627",
  "variantId": "5_14709746_A_G",
  "variantRsId": "rs1224951926",
  "variantFromSourceId": "VCV000906120",
  "cohortPhenotypes": [
    "Craniometaphyseal dysplasia, autosomal dominant"
  ],
  "diseaseFromSource": "Craniometaphyseal dysplasia, autosomal dominant",
  "diseaseFromSourceId": "C1852502",
  "diseaseFromSourceMappedId": "MONDO_0015465",
  "variantHgvsId": "NC_000005.10:g.14709746A>G"
}
'''

In [17]:
unmapped_apr_rcvs = set()
unmapped_apr_traits = Counter()
with gzip.open(apr_evidence_path, 'rt') as f:
    for line in f:
        if 'diseaseFromSourceMappedId' not in line:
            m = re.search('"studyId": "([^"]+)"', line)
            if m and m.group(1):
                unmapped_apr_rcvs.add(m.group(1))
            m = re.search('"diseaseFromSource": "([^"]+)"', line)
            if m and m.group(1):
                unmapped_apr_traits[m.group(1)] += 1

In [18]:
len(unmapped_apr_rcvs)

696713

In [39]:
len(unmapped_apr_traits)

8931

In [20]:
for trait, count in sorted(unmapped_apr_traits.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(trait, ':', count)

Inborn genetic diseases : 358144
Ovarian serous cystadenocarcinoma : 51925
Cardiomyopathy : 20524
Acute myeloid leukemia : 17551
Sarcoma : 16094
Hypertrophic cardiomyopathy : 13143
Hepatocellular carcinoma : 12693
Thymoma : 12003
Melanoma : 11312
Uterine carcinosarcoma : 8651


In [21]:
mapped_jan_rcvs = set()
with gzip.open(jan_evidence_path, 'rt') as f:
    for line in f:
        if 'diseaseFromSourceMappedId' in line:
            m = re.search('"studyId": "([^"]+)"', line)
            if m and m.group(1):
                mapped_jan_rcvs.add(m.group(1))

In [22]:
rcvs_mapped_in_jan_unmapped_in_apr = unmapped_apr_rcvs.intersection(mapped_jan_rcvs)

In [23]:
len(rcvs_mapped_in_jan_unmapped_in_apr)

594647

In [24]:
def view_set(s, lim=10):
    n = 0
    for x in s:
        print(x)
        n += 1
        if n >= lim:
            break

In [25]:
view_set(rcvs_mapped_in_jan_unmapped_in_apr)

RCV003039263
RCV005944006
RCV006007306
RCV005758393
RCV006160161
RCV002291609
RCV006041600
RCV002494055
RCV005652783
RCV002835641


RCV003039263 : trait is "Landau-Kleffner syndrome" which is present in the January mappings and missing in the April mappings. The previous mapping was http://www.ebi.ac.uk/efo/EFO_1001010 which is replaced by a Mondo term http://purl.obolibrary.org/obo/MONDO_0009509. It does not show up in the March curation but does appear (with the obsolete EFO term) in automated mappings!

In [32]:
from cmat.trait_mapping.ols import is_current_and_in_ontology, get_replacement_term, get_label_and_synonyms_from_ols

In [28]:
is_current_and_in_ontology('http://www.ebi.ac.uk/efo/EFO_1001010', 'efo')

False

Automated mappings were generated on 17 March, EFO release with the mass deprecations was on [16 March](https://github.com/EBISPOT/efo/releases/tag/v3.88.0). Given the [previous mapping processing](https://github.com/EBIvariation/CMAT/blob/324b96086664748fef90e5b4f45299cf948fa03d/cmat/trait_mapping/trait.py#L78) which includes checking `is_current_and_in_ontology`, I don't see how an obsolete mapping could have made it into automated mappings. My best guess for what happened is that EFO was updated but the newer version was not yet indexed in OLS, so we didn't catch the obsoletes for the curation.

My suggestion would be to do an ad-hoc curation of the 3k obsolete terms & their replacements and add these to latest mappings. Then re-run evidence generation and re-submit

In [37]:
obsolete_mappings_file = 'obsolete_mappings.tsv'
obsolete_for_curation_file = 'obsolete_for_curation_filtered.tsv'

In [42]:
with open(obsolete_mappings_file) as in_file, open(obsolete_for_curation_file, 'w+') as out_file:
    out_file.write('trait_name\tobsolete_uri\tobsolete_label\treplacement_uri\treplacement_label\n')
    for line in in_file:
        trait_name, obsolete_uri, obsolete_label = (x.strip() for x in line.split('\t'))
        if trait_name.lower() in unmapped_apr_traits:
            replacement_uri = get_replacement_term(obsolete_uri)
            replacement_label, _ = get_label_and_synonyms_from_ols(replacement_uri)
            out_file.write(f'{trait_name}\t{obsolete_uri}\t{obsolete_label}\t{replacement_uri}\t{replacement_label}\n')

In [40]:
view_set(unmapped_apr_traits)

Primary dilated cardiomyopathy
Renal cell carcinoma
Epilepsy
Hypertrophic cardiomyopathy
Birt-Hogg-Dube syndrome
Idiopathic generalized epilepsy
RASopathy
Parathyroid carcinoma
Pyruvate carboxylase deficiency
Alpha-methylacyl-CoA racemase deficiency


In [41]:
unmapped_apr_traits = {x.lower() for x in unmapped_apr_traits}